# Reality2Unity — Week 1 Baseline (Colab)

Google Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서부터 실행하세요.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/mir0173/Reality2Unity.git"
BRANCH = "develop"
REPO_DIR = Path("/content/Reality2Unity")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

required = [
    "python/main.py",
    "python/depth_estimation/midas.py",
    "python/terrain_generation/legacy_converter.py",
]


## 실행 패키지 준비



In [ ]:
%pip -q install timm opencv-python-headless matplotlib


In [ ]:
%matplotlib inline
import sys

python_dir = str(REPO_DIR / "python")
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)

from main import main


## 입력 이미지 업로드



In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

image_path = Path('/content/drive/MyDrive/Reality2Unity/test.png')

print("Image:", image_path)

## Baseline 실행



In [ ]:
main(str(image_path))


# Reality2Unity — Week 2 (Colab)

## 깊이맵 및 이미지

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from depth_estimation.midas import load_midas_model, estimate_depth

midas, transform, device = load_midas_model()

depth = estimate_depth(str(image_path), midas, transform, device)

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

## 	초점거리 비교

In [ ]:

H, W = depth.shape

depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
Z = 1 / (depth_norm + 0.15)

step = 2

u, v = np.meshgrid(
    np.arange(0, W, step),
    np.arange(0, H, step)
)

z = Z[::step, ::step]
rgb = image[::step, ::step] / 255.0
mask = depth_norm[::step, ::step] > 0.1

cx = (W - 1) / 2
cy = (H - 1) / 2

fig = plt.figure(figsize=(18, 6))

for i, scale in enumerate([0.5, 1.0, 2.0]):
    fx = W * scale
    fy = W * scale

    x = (u - cx) * z / fx
    y = -(v - cy) * z / fy

    ax = fig.add_subplot(1, 3, i + 1, projection='3d')

    ax.scatter(
        x[mask],
        y[mask],
        z[mask],
        c=rgb[mask],
        s=1
    )

    ax.view_init(elev=20, azim=-70)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 2)
    ax.set_zlim(0, 7)
    ax.set_title(f'Focal Length = {scale}W')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    print(f'Focal Length: {scale}W')
    print(f'X range: {np.ptp(x[mask]):.3f}')
    print(f'Y range: {np.ptp(y[mask]):.3f}')

plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(18, 6))

fx = W
fy = W

for i, offset in enumerate([0.05, 0.15, 0.5]):
    Z = 1 / (depth_norm + offset)
    z = Z[::step, ::step]

    x = (u - cx) * z / fx
    y = -(v - cy) * z / fy

    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    ax.scatter(x[mask], y[mask], z[mask], c=rgb[mask], s=1)

    ax.view_init(elev=20, azim=-70)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 2)
    ax.set_zlim(0, 7)

    ax.set_title(f'Depth Offset = {offset}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    print(f'Offset: {offset}')
    print(f'Z range: {np.ptp(z[mask]):.3f}')

plt.tight_layout()
plt.show()

## Point Cloud 저장

In [ ]:

from google.colab import files

H, W = depth.shape

depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
Z = 1 / (depth_norm + 0.15)

step = 2

u, v = np.meshgrid(
    np.arange(0, W, step),
    np.arange(0, H, step)
)

z = Z[::step, ::step]

fx = W
fy = W
cx = (W - 1) / 2
cy = (H - 1) / 2

x = (u - cx) * z / fx
y = -(v - cy) * z / fy

mask = depth_norm[::step, ::step] > 0.1

points = np.column_stack((x[mask], y[mask], z[mask]))
colors = image[::step, ::step][mask]

output_path = "/content/drive/MyDrive/Reality2Unity/pointcloud_baseline.ply"

with open(output_path, "w") as f:
    f.write("ply\n")
    f.write("format ascii 1.0\n")
    f.write(f"element vertex {len(points)}\n")
    f.write("property float x\n")
    f.write("property float y\n")
    f.write("property float z\n")
    f.write("property uchar red\n")
    f.write("property uchar green\n")
    f.write("property uchar blue\n")
    f.write("end_header\n")

    for p, c in zip(points, colors):
        f.write(f"{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n")

print("Saved:", output_path)
print("Number of points:", len(points))


## 저장 전 출력 정리

실행 후 Git에 반영하기 전에는 Colab의 **수정 → 모든 출력 지우기**를 실행하세요. Colab 설정에서 **저장 시 코드 셀 출력 생략**을 활성화하세요.